In [ ]:
on["PATH"] = JAVA_HOME + r"\bin;" + os.environ["PATH"]


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_04"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Função de exportação
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))

from utils.export_utils import exportar_csv


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# GOLD 04 - ADOÇÃO DE TECNOLOGIAS
# ---------------------------------------------------------------------

caminho_gold_04 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_04_adocao_tecnologias"

arquivos_gold_04 = [
    str(arquivo) for arquivo in caminho_gold_04.glob("part-*.csv")
]

if not arquivos_gold_04:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_04}"
    )

df_tecnologias = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_04)
)


# ---------------------------------------------------------------------
# NOMES DAS CATEGORIAS
# ---------------------------------------------------------------------
"""
Os nomes técnicos das categorias são traduzidos para rótulos mais legíveis sem alterar os valores originais usados nos filtros. Isso facilita a leitura dos rankings e dos arquivos exportados.
"""

df_tecnologias = (
    df_tecnologias.withColumn(
        "categoria_nome",
        F.when(
            F.col("categoria") == "linguagem_de_programacao",
            "Linguagens de Programação"
        )
        .when(
            F.col("categoria") == "ferramenta_de_bi",
            "Ferramentas de BI"
        )
        .when(
            F.col("categoria") == "cloud",
            "Cloud"
        )
        .when(
            F.col("categoria") == "banco_de_dados",
            "Bancos de Dados"
        )
        .when(
            F.col("categoria") == "ferramenta_etl_data_engineer",
            "ETL - Data Engineer"
        )
        .when(
            F.col("categoria") == "ferramenta_etl_data_analyst",
            "ETL - Data Analyst"
        )
        .when(
            F.col("categoria") == "linguagem_preferida_2025_2026",
            "Linguagem Preferida"
        )
        .otherwise(F.col("categoria"))
    )
)


# ---------------------------------------------------------------------
# BASE ANALÍTICA
# ---------------------------------------------------------------------

# Adoção igual a zero não entra nos rankings e comparações históricas
"""
O recorte considera somente opções com adoção maior que zero. Assim, rankings e comparações históricas são construídos a partir de tecnologias efetivamente selecionadas, evitando que registros sem adoção ocupem posições ou sejam tratados como movimento histórico.
"""
df_adocao_valida = (
    df_tecnologias.filter(F.col("pct_adocao") > 0)
)


# ---------------------------------------------------------------------
# COBERTURA DAS CATEGORIAS POR EDIÇÃO
# ---------------------------------------------------------------------
"""
A cobertura é verificada antes das comparações porque as categorias não estão disponíveis da mesma forma nas três pesquisas. Nos outputs, Cloud e Ferramentas de BI aparecem nas três edições; Bancos de Dados e as duas categorias de ETL aparecem em duas; Linguagens de Programação em 2023-2024 e 2024-2025; e Linguagem Preferida somente em 2025-2026.
"""

cobertura_categorias = (
    df_adocao_valida
    .groupBy("categoria", "categoria_nome")
    .agg(
        F.countDistinct("edicao").alias("qtd_edicoes"),
        F.concat_ws(
            ", ",
            F.sort_array(F.collect_set("edicao"))
        ).alias("edicoes")
    )
    .orderBy(F.desc("qtd_edicoes"), "categoria_nome")
)

print("\n" + "=" * 120)
print("01. COBERTURA DAS CATEGORIAS POR EDIÇÃO")
print("=" * 120)

cobertura_categorias.show(100, truncate=False)


# ---------------------------------------------------------------------
# RANKING DE ADOÇÃO POR CATEGORIA E EDIÇÃO
# ---------------------------------------------------------------------
"""
O ranking é calculado separadamente por edição e categoria e usa pct_adocao como critério principal. Esse percentual é mais adequado do que a contagem absoluta porque o número de elegíveis varia entre categorias e edições; selecionaram é usado como desempate.
"""

janela_ranking = (
    Window
    .partitionBy("edicao", "categoria")
    .orderBy(F.desc("pct_adocao"), F.desc("selecionaram"))
)

ranking_adocao = (
    df_adocao_valida
    .withColumn(
        "ranking",
        F.row_number().over(janela_ranking)
    )
    .select(
        "edicao",
        "categoria",
        "categoria_nome",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "categoria_nome", "ranking")
)

print("\n" + "=" * 120)
print("02. TOP 10 TECNOLOGIAS POR CATEGORIA E EDIÇÃO")
print("=" * 120)

(
    ranking_adocao
    .filter(F.col("ranking") <= 10)
    .show(300, truncate=False)
)


# ---------------------------------------------------------------------
# TOP 5 DA EDIÇÃO MAIS RECENTE
# ---------------------------------------------------------------------
"""
O recorte da edição mais recente resume as tecnologias líderes em cada categoria disponível em 2025-2026. Nos outputs, PostgreSQL lidera Bancos de Dados com 36,8%, AWS lidera Cloud com 48,3%, Power BI lidera BI com 59,0% e scripts Python lideram ETL tanto para Data Analyst quanto para Data Engineer.
"""

ultima_edicao = (
    df_tecnologias
    .agg(F.max("edicao").alias("ultima_edicao"))
    .first()["ultima_edicao"]
)

top_5_ultima_edicao = (
    ranking_adocao
    .filter(
        (F.col("edicao") == ultima_edicao)
        & (F.col("ranking") <= 5)
    )
    .orderBy("categoria_nome", "ranking")
)

print("\n" + "=" * 120)
print(f"03. TOP 5 POR CATEGORIA - {ultima_edicao}")
print("=" * 120)

top_5_ultima_edicao.show(100, truncate=False)


# ---------------------------------------------------------------------
# LINGUAGENS DE PROGRAMAÇÃO - HISTÓRICO
# ---------------------------------------------------------------------
"""
A categoria de Linguagens de Programação possui duas edições comparáveis. SQL permanece na primeira posição, enquanto Python passa de 74,9% em 2023-2024 para 81,8% em 2024-2025; por isso, essa série não é estendida artificialmente até 2025-2026.
"""

linguagens_historico = (
    ranking_adocao
    .filter(F.col("categoria") == "linguagem_de_programacao")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "ranking")
)

print("\n" + "=" * 120)
print("04. LINGUAGENS DE PROGRAMAÇÃO - HISTÓRICO")
print("=" * 120)

(
    linguagens_historico
    .filter(F.col("ranking") <= 10)
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# FERRAMENTAS DE BI - HISTÓRICO
# ---------------------------------------------------------------------
"""
Ferramentas de BI permitem comparação nas três edições. Power BI permanece na liderança, com 57,1% em 2023-2024 e 59,0% em 2025-2026, enquanto outras ferramentas apresentam trajetórias distintas ao longo do período.
"""

bi_historico = (
    ranking_adocao
    .filter(F.col("categoria") == "ferramenta_de_bi")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "ranking")
)

print("\n" + "=" * 120)
print("05. FERRAMENTAS DE BI - HISTÓRICO")
print("=" * 120)

(
    bi_historico
    .filter(F.col("ranking") <= 10)
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# CLOUD - HISTÓRICO
# ---------------------------------------------------------------------
"""
A categoria Cloud está presente nas três edições, mas nem todas as opções aparecem desde o início. Em 2023-2024, o output disponível registra apenas a opção de servidores on-premise; por isso, a comparação posterior usa a primeira e a última edição válida de cada tecnologia, e não força uma mesma janela temporal para todas.
"""

cloud_historico = (
    ranking_adocao
    .filter(F.col("categoria") == "cloud")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "ranking")
)

print("\n" + "=" * 120)
print("06. CLOUD - HISTÓRICO")
print("=" * 120)

(
    cloud_historico
    .filter(F.col("ranking") <= 10)
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# BANCOS DE DADOS - HISTÓRICO
# ---------------------------------------------------------------------
"""
Bancos de Dados possui dados comparáveis apenas em 2024-2025 e 2025-2026. PostgreSQL mantém a liderança e passa de 31,4% para 36,8%, enquanto o ranking também mostra avanço de SQL Server, S3 e MySQL.
"""

bancos_historico = (
    ranking_adocao
    .filter(F.col("categoria") == "banco_de_dados")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "ranking")
)

print("\n" + "=" * 120)
print("07. BANCOS DE DADOS - HISTÓRICO")
print("=" * 120)

(
    bancos_historico
    .filter(F.col("ranking") <= 10)
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# ETL - DATA ENGINEER
# ---------------------------------------------------------------------
"""
Para Data Engineer, scripts Python permanecem como a opção mais adotada e avançam de 81,9% para 86,6%. SQL & stored procedures também se mantém em nível elevado, enquanto ferramentas específicas aparecem com adoções menores.
"""

etl_engineer = (
    ranking_adocao
    .filter(F.col("categoria") == "ferramenta_etl_data_engineer")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "ranking")
)

print("\n" + "=" * 120)
print("08. ETL - DATA ENGINEER")
print("=" * 120)

(
    etl_engineer
    .filter(F.col("ranking") <= 10)
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# ETL - DATA ANALYST
# ---------------------------------------------------------------------
"""
Para Data Analyst, scripts Python lideram nas duas edições e aumentam de 50,6% para 57,9%. SQL & stored procedures também cresce, de 48,0% para 53,3%, reforçando a predominância dessas duas abordagens no recorte.
"""

etl_analyst = (
    ranking_adocao
    .filter(F.col("categoria") == "ferramenta_etl_data_analyst")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("edicao", "ranking")
)

print("\n" + "=" * 120)
print("09. ETL - DATA ANALYST")
print("=" * 120)

(
    etl_analyst
    .filter(F.col("ranking") <= 10)
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# LINGUAGEM PREFERIDA - 2025-2026
# ---------------------------------------------------------------------
"""
Esta categoria existe somente em 2025-2026 e, por isso, é tratada como retrato atual e não como série histórica. Python aparece com 92,0% e SQL com 84,2%, muito acima das demais opções listadas.
"""

linguagem_preferida = (
    ranking_adocao
    .filter(F.col("categoria") == "linguagem_preferida_2025_2026")
    .select(
        "edicao",
        "ranking",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy("ranking")
)

print("\n" + "=" * 120)
print("10. LINGUAGEM PREFERIDA - 2025-2026")
print("=" * 120)

linguagem_preferida.show(100, truncate=False)


# ---------------------------------------------------------------------
# TECNOLOGIAS COMPARÁVEIS ENTRE EDIÇÕES
# ---------------------------------------------------------------------

# Tecnologias com adoção > 0 em pelo menos duas edições
"""
Uma tecnologia só entra no comparativo de evolução quando possui adoção positiva em pelo menos duas edições. O critério evita calcular variação a partir de uma única observação e respeita as diferenças de cobertura identificadas anteriormente.
"""
tecnologias_comparaveis = (
    df_adocao_valida
    .groupBy("categoria", "categoria_nome", "opcao")
    .agg(F.countDistinct("edicao").alias("qtd_edicoes"))
    .filter(F.col("qtd_edicoes") >= 2)
)


# ---------------------------------------------------------------------
# PRIMEIRA E ÚLTIMA EDIÇÃO VÁLIDA DE CADA TECNOLOGIA
# ---------------------------------------------------------------------
"""
Como a disponibilidade das tecnologias varia entre as pesquisas, a evolução é medida entre a primeira e a última edição em que cada opção possui adoção válida. Assim, a análise não pressupõe que todas as tecnologias existiam ou eram perguntadas desde 2023-2024.
"""

janela_inicio = (
    Window
    .partitionBy("categoria", "opcao")
    .orderBy(F.asc("edicao"))
)

janela_fim = (
    Window
    .partitionBy("categoria", "opcao")
    .orderBy(F.desc("edicao"))
)

primeira_adocao = (
    df_adocao_valida
    .withColumn(
        "rn",
        F.row_number().over(janela_inicio)
    )
    .filter(F.col("rn") == 1)
    .select(
        "categoria",
        "opcao",
        F.col("edicao").alias("primeira_edicao"),
        F.col("pct_adocao").alias("pct_inicio")
    )
)

ultima_adocao = (
    df_adocao_valida
    .withColumn(
        "rn",
        F.row_number().over(janela_fim)
    )
    .filter(F.col("rn") == 1)
    .select(
        "categoria",
        "opcao",
        F.col("edicao").alias("ultima_edicao"),
        F.col("pct_adocao").alias("pct_fim")
    )
)


# ---------------------------------------------------------------------
# EVOLUÇÃO DAS TECNOLOGIAS COMPARÁVEIS
# ---------------------------------------------------------------------
"""
A variação é expressa em pontos percentuais entre a primeira e a última observação válida de cada tecnologia. Esse desenho permite comparar direção e intensidade do movimento sem confundir mudança de adoção com diferenças no número de elegíveis.
"""

comparativo_evolucao = (
    tecnologias_comparaveis
    .join(
        primeira_adocao,
        ["categoria", "opcao"],
        "left"
    )
    .join(
        ultima_adocao,
        ["categoria", "opcao"],
        "left"
    )
    .withColumn(
        "variacao_pp",
        F.round(F.col("pct_fim") - F.col("pct_inicio"), 1)
    )
    .select(
        "categoria",
        "categoria_nome",
        "opcao",
        "primeira_edicao",
        "ultima_edicao",
        "qtd_edicoes",
        "pct_inicio",
        "pct_fim",
        "variacao_pp"
    )
    .orderBy("categoria_nome", F.desc("variacao_pp"))
)

print("\n" + "=" * 120)
print("11. EVOLUÇÃO DAS TECNOLOGIAS COMPARÁVEIS")
print("=" * 120)

comparativo_evolucao.show(300, truncate=False)


# ---------------------------------------------------------------------
# MAIORES CRESCIMENTOS DE ADOÇÃO
# ---------------------------------------------------------------------
"""
Os crescimentos são ranqueados dentro de cada categoria, mantendo apenas variações positivas. Entre os movimentos observados, scripts Python em ETL para Data Analyst cresce 7,3 p.p.; Python em Linguagens de Programação cresce 6,9 p.p.; SQLite cresce 6,0 p.p. em Bancos de Dados; e AWS cresce 4,2 p.p. em Cloud.
"""

janela_crescimento = (
    Window
    .partitionBy("categoria")
    .orderBy(F.desc("variacao_pp"))
)

maiores_crescimentos = (
    comparativo_evolucao
    .filter(F.col("variacao_pp") > 0)
    .withColumn(
        "ranking_crescimento",
        F.row_number().over(janela_crescimento)
    )
    .filter(F.col("ranking_crescimento") <= 5)
    .select(
        "categoria",
        "categoria_nome",
        "ranking_crescimento",
        "opcao",
        "primeira_edicao",
        "ultima_edicao",
        "pct_inicio",
        "pct_fim",
        "variacao_pp"
    )
    .orderBy("categoria_nome", "ranking_crescimento")
)

print("\n" + "=" * 120)
print("12. MAIORES CRESCIMENTOS DE ADOÇÃO")
print("=" * 120)

maiores_crescimentos.show(100, truncate=False)


# ---------------------------------------------------------------------
# MAIORES QUEDAS DE ADOÇÃO
# ---------------------------------------------------------------------
"""
As quedas são ranqueadas separadamente para não misturar crescimento e retração no mesmo destaque. Nos outputs, Tableau apresenta a maior queda em BI, de 19,0% para 13,6% (-5,4 p.p.), e Apache Airflow recua 2,8 p.p. no recorte de ETL para Data Analyst.
"""

janela_queda = (
    Window
    .partitionBy("categoria")
    .orderBy(F.asc("variacao_pp"))
)

maiores_quedas = (
    comparativo_evolucao
    .filter(F.col("variacao_pp") < 0)
    .withColumn(
        "ranking_queda",
        F.row_number().over(janela_queda)
    )
    .filter(F.col("ranking_queda") <= 5)
    .select(
        "categoria",
        "categoria_nome",
        "ranking_queda",
        "opcao",
        "primeira_edicao",
        "ultima_edicao",
        "pct_inicio",
        "pct_fim",
        "variacao_pp"
    )
    .orderBy("categoria_nome", "ranking_queda")
)

print("\n" + "=" * 120)
print("13. MAIORES QUEDAS DE ADOÇÃO")
print("=" * 120)

maiores_quedas.show(100, truncate=False)


# ---------------------------------------------------------------------
# EXPORTAÇÃO DOS OUTPUTS
# ---------------------------------------------------------------------

exportar_csv(
    cobertura_categorias,
    OUTPUT_DIR,
    "cobertura_categorias.csv"
)

exportar_csv(
    ranking_adocao,
    OUTPUT_DIR,
    "ranking_adocao_por_categoria.csv"
)

exportar_csv(
    top_5_ultima_edicao,
    OUTPUT_DIR,
    "top_5_tecnologias_ultima_edicao.csv"
)

exportar_csv(
    linguagens_historico,
    OUTPUT_DIR,
    "linguagens_programacao_historico.csv"
)

exportar_csv(
    bi_historico,
    OUTPUT_DIR,
    "ferramentas_bi_historico.csv"
)

exportar_csv(
    cloud_historico,
    OUTPUT_DIR,
    "cloud_historico.csv"
)

exportar_csv(
    bancos_historico,
    OUTPUT_DIR,
    "bancos_dados_historico.csv"
)

exportar_csv(
    etl_engineer,
    OUTPUT_DIR,
    "etl_data_engineer_historico.csv"
)

exportar_csv(
    etl_analyst,
    OUTPUT_DIR,
    "etl_data_analyst_historico.csv"
)

exportar_csv(
    linguagem_preferida,
    OUTPUT_DIR,
    "linguagem_preferida_2025_2026.csv"
)

exportar_csv(
    comparativo_evolucao,
    OUTPUT_DIR,
    "evolucao_tecnologias_comparaveis.csv"
)

exportar_csv(
    maiores_crescimentos,
    OUTPUT_DIR,
    "maiores_crescimentos_adocao.csv"
)

exportar_csv(
    maiores_quedas,
    OUTPUT_DIR,
    "maiores_quedas_adocao.csv"
)